# 02 — Feature engineering

Inspect the canonical feature table and demonstrate the full proposal feature
set on a small event-level example.

Definitions live in `docs/FEATURES.md` and `src/focustrack/features/`.

In [1]:
from pathlib import Path

import pandas as pd

from focustrack.config import load_config

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
cfg = load_config(ROOT / "configs" / "c1_proposed.yaml")

features = pd.read_csv(ROOT / "data" / "processed" / "features.csv")
features.shape

(6600, 33)

In [2]:
native = [
    "typing_rate",
    "inter_key_cv",
    "backspace_ratio",
    "idle_ratio",
    "mouse_speed_px_s",
    "switch_rate",
]
features.groupby(["study_mode", "task"])[native].mean().round(2)

typing_rate  inter_key_cv  backspace_ratio  \
study_mode task                                                      
non-screen handwriting         1.82          0.61             0.13   
           reading             1.77          0.64             0.13   
screen     assignment         41.14          0.56             0.12   
           coding             50.08          0.55             0.12   
           writing            65.25          0.58             0.12   
           zoom                8.79          0.54             0.12   

                        idle_ratio  mouse_speed_px_s  switch_rate  
study_mode task                                                    
non-screen handwriting        0.94             11.51         0.19  
           reading            0.94             11.51         0.19  
screen     assignment         0.54            112.88         2.44  
           coding             0.53            111.91         2.39  
           writing            0.56            107.41         2.53  
           zoom               0.54            160.28         2.46

## The full feature set needs event-level logs

`build_from_events` computes all 14 features from key/mouse/application event
metadata. The example below is one minute with six keystrokes, a mouse move, a
click and one application switch.

In [3]:
import numpy as np

from focustrack.features.build import build_from_events

sessions = pd.DataFrame(
    {
        "session_id": ["demo"],
        "participant_id": ["p01"],
        "study_mode": ["screen"],
        "task": ["coding"],
        "duration_seconds": [60],
    }
)
events = pd.DataFrame(
    {
        "session_id": ["demo"] * 9,
        "t": [0, 1, 2, 3, 4, 5, 10, 11, 12],
        "kind": ["key"] * 6 + ["move", "click", "switch"],
        "dist": [np.nan] * 6 + [20.0, np.nan, np.nan],
        "app": [None] * 8 + ["Visual Studio Code"],
    }
)
full = build_from_events(events, sessions, cfg)
full[[
    "typing_rate", "inter_key_mean_ms", "inter_key_cv", "backspace_ratio",
    "burst_count", "mean_burst_length", "mouse_path_px", "click_rate",
    "idle_ratio", "switch_rate", "unique_apps", "app_category_entropy",
]]

,typing_rate,inter_key_mean_ms,inter_key_cv,backspace_ratio,burst_count,mean_burst_length,mouse_path_px,click_rate,idle_ratio,switch_rate,unique_apps,app_category_entropy
0,6.0,1000.0,NaN,NaN,1.0,6.0,20.0,1.0,0.85,1.0,1.0,-0.0


## Before modelling

- Do not fill missing values with constants here; impute inside the fold.
- Check feature distributions separately for screen and non-screen sessions.
- Keep identifiers, probe ratings and labels out of the feature matrix.